# Overlay detector — full pipeline on Colab Pro

Two-stream (Bayar/SRM forensic noise + RGB backbone) face-region fraud detector.
Runs the full pipeline: **data download -> face-crop caching -> train -> infer -> submit**.

**Colab Pro** recommended: A100/V100 GPU, longer runtime, more RAM.
Also works on free-tier T4 (reduce `batch_size` to 16 if you hit OOM).

---

### Two ways to use this notebook

**Option A — Run directly in Colab:** Upload this `.ipynb` to Colab, set runtime to GPU, run all cells top-to-bottom.

**Option B — IDE remote kernel:** Follow the ngrok tunnel setup from `run_baseline.ipynb` STEP 1,
then connect this notebook's kernel to the Colab server and run STEP 2+ cells.

---

## STEP 1 — Environment setup

In [ ]:
# [1] GPU check + clone + install
import torch, os

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f'GPU: {gpu} ({vram:.1f} GB)')
else:
    raise RuntimeError('No GPU — set Runtime > Change runtime type > GPU (A100 for Colab Pro)')

if not os.path.exists('/content/kaggle_freuid_2026'):
    !git clone --depth 1 -b overlay-detector https://github.com/hyunsoochung-portfolio/kaggle_freuid_2026.git /content/kaggle_freuid_2026

%cd /content/kaggle_freuid_2026
!pip install -q -e .   # reuses Colab's built-in torch (CUDA); installs timm, facenet-pytorch, etc.

In [ ]:
# [2] Kaggle auth — uses Colab Secrets (KAGGLE_USERNAME, KAGGLE_KEY)
# Add these in the Colab sidebar: key icon > Add a new secret
import os, json
from google.colab import userdata

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({'username': userdata.get('KAGGLE_USERNAME'),
               'key': userdata.get('KAGGLE_KEY')}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('kaggle.json written')

In [ ]:
# [3] Download competition data
# You must have accepted the competition rules on the Kaggle website first (403 otherwise)
import os
if not os.path.exists('data/train/train'):
    !kaggle competitions download -c the-freuid-challenge-2026-ijcai-ecai -p data
    !cd data && unzip -q -o '*.zip' && rm -f *.zip
else:
    print('data/ already exists, skipping download')

!echo 'train images:' && ls data/train/train | wc -l
!echo 'test images:'  && ls data/public_test/public_test | wc -l

In [ ]:
# [4] Mount Google Drive — checkpoints + submissions auto-save to Drive
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/drive/MyDrive/freuid/checkpoints', exist_ok=True)
os.makedirs('/content/drive/MyDrive/freuid/submissions', exist_ok=True)

# Symlink so train.py / infer.py saves go to Drive automatically
!rm -rf checkpoints submissions
!ln -s /content/drive/MyDrive/freuid/checkpoints checkpoints
!ln -s /content/drive/MyDrive/freuid/submissions submissions
print('checkpoints/, submissions/ -> linked to Drive')

## STEP 2 — Configure the overlay run

The overlay detector needs a Colab-tuned config. Adjust `batch_size` and `epochs` below
based on your GPU:

| GPU | VRAM | Recommended `batch_size` |
|-----|------|--------------------------|
| A100 | 40 GB | 64 |
| V100 | 16 GB | 32 |
| T4 | 16 GB | 16 |

In [ ]:
%%writefile configs/overlay_colab.yaml
# Overlay detector — Colab Pro config (A100/V100)
name: overlay_colab
seed: 42

data_dir: data
image_size: 224
val_fraction: 0.1
epochs: 5
batch_size: 16
lr: 0.0001
weight_decay: 0.0001
num_workers: 0   # MTCNN singleton is not fork-safe

model_type: overlay

model:
  type: overlay
  noise_frontend: bayar
  noise_feat_dim: 128
  use_rgb_stream: true
  rgb_backbone: resnet34
  rgb_pretrained: true
  fusion_dim: 128

overlay:
  crop_margin: 0.75
  crop_cache_dir: data/processed/overlay_crops

## STEP 3 — Pre-cache face crops

MTCNN detects faces and saves cropped regions to disk. This is a one-time cost
(~30 min on A100 for the full dataset). Subsequent runs reuse the cache.

If the session disconnects mid-crop, re-running skips already-cached images.

In [ ]:
from freuid.config import load_config
from freuid.data import precache_crops

cfg = load_config('configs/overlay_colab.yaml')
precache_crops(cfg)

## STEP 4 — Train

In [ ]:
!python -m freuid.train --config configs/overlay_colab.yaml

## STEP 5 — Inference

In [ ]:
CKPT = 'checkpoints/overlay_colab.pt'
OUT  = 'submissions/overlay_colab.csv'

!python -m freuid.infer --checkpoint {CKPT} --config configs/overlay_colab.yaml --out {OUT}
!head -3 {OUT} && echo '---' && wc -l {OUT}

## STEP 6 — Submit to Kaggle

In [ ]:
OUT = 'submissions/overlay_colab.csv'
!kaggle competitions submit -c the-freuid-challenge-2026-ijcai-ecai -f {OUT} -m 'overlay detector (colab pro)'